# Train sp1DLC with the mobile label export

Set **Runtime > Change runtime type > GPU**, then run the cells in order. This notebook uses the frozen September 24, 2026 label export. It creates a new dated project copy in Google Drive, trains the existing **19-body-part** DLC project with the updated seven focus labels, and never overwrites the source project or website. Training checkpoints are written to the new Drive copy.

The source Drive project has extracted frame images but no original MP4 videos. This notebook trains and evaluates; video analysis is a separate later step. DeepLabCut's [official Colab guide](https://deeplabcut.github.io/DeepLabCut/examples/COLAB/COLAB_YOURDATA_TrainNetwork_VideoAnalysis.html) uses the same training and evaluation APIs.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--pre', 'deeplabcut'])
print('If Colab requests a runtime restart, restart it now and continue with the next cell.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import deeplabcut, torch
assert torch.cuda.is_available(), 'No GPU is active. Choose Runtime > Change runtime type > GPU.'
print('DeepLabCut:', deeplabcut.__version__)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

MY_DRIVE = Path('/content/drive/MyDrive')
if not MY_DRIVE.is_dir():
    MY_DRIVE = Path('/content/drive/My Drive')
SOURCE = MY_DRIVE / 'sp1DLC' / 'sp1DLC'
EXPORT = MY_DRIVE / 'sp1DLC' / 'MobileDLC_Colab_Export_2026-09-24'
LABELS_ZIP = EXPORT / 'colab-complete-labels-20260924-230130.zip'
SNAPSHOT = EXPORT / 'colab-labels-20260924-230130.json'
RUNS = MY_DRIVE / 'sp1DLC' / 'Colab_Training_Runs'
RUN_NAME = 'sp1DLC_mobile_20260924_230130_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
PROJECT = RUNS / RUN_NAME
CONFIG = PROJECT / 'config.yaml'
for path in (SOURCE / 'config.yaml', SOURCE / 'labeled-data', LABELS_ZIP, SNAPSHOT):
    assert path.exists(), f'Missing Drive input: {path}'
assert not PROJECT.exists(), f'Run folder already exists: {PROJECT}'
print('Source project:', SOURCE)
print('Frozen label ZIP:', LABELS_ZIP)
print('New training copy:', PROJECT)


In [ ]:
import json, re, shutil, zipfile
import pandas as pd
import yaml
from pathlib import PurePosixPath

RUNS.mkdir(parents=True, exist_ok=True)
PROJECT.mkdir(exist_ok=False)
shutil.copy2(SOURCE / 'config.yaml', CONFIG)
shutil.copytree(SOURCE / 'labeled-data', PROJECT / 'labeled-data')
with zipfile.ZipFile(LABELS_ZIP) as archive:
    names = archive.namelist()
    pattern = re.compile(r'labeled-data/Test [0-9]+/CollectedData_Kobe\.(h5|csv)')
    assert len(names) == 62 and len(set(names)) == 62
    assert all(pattern.fullmatch(name) and len(PurePosixPath(name).parts) == 3 for name in names)
    archive.extractall(PROJECT)

with CONFIG.open(encoding='utf-8') as handle:
    cfg = yaml.safe_load(handle)
assert cfg['scorer'] == 'Kobe' and len(cfg['bodyparts']) == 19
assert len(cfg['video_sets']) == 31 and cfg['engine'] == 'pytorch'
cfg['project_path'] = str(PROJECT)
cfg['pose_config_path'] = None
cfg['iteration'] = 4
cfg['video_sets'] = {
    str(PROJECT / 'videos' / PurePosixPath(str(old).replace('\\', '/')).name): value
    for old, value in cfg['video_sets'].items()
}
with CONFIG.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

labeled = PROJECT / 'labeled-data'
images = [p for p in labeled.rglob('*') if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}]
folders = [p for p in labeled.iterdir() if p.is_dir()]
assert len(images) == 1865 and len(folders) == 31, (len(images), len(folders))
assert len(list(labeled.glob('*/CollectedData_Kobe.h5'))) == 31
snapshot = json.loads(SNAPSHOT.read_text(encoding='utf-8'))
assert snapshot['schema'] == 'focus7' and snapshot['scorer'] == 'Kobe'
assert len(snapshot['frames']) == 1865
focus_parts = ('nose', 'left_ear', 'right_ear', 'spine_mid', 'left_hip', 'right_hip', 'tail_base')
tables = {folder.name: pd.read_hdf(folder / 'CollectedData_Kobe.h5') for folder in folders}
complete = 0
for frame in snapshot['frames']:
    key = frame['key']
    _, folder, image_name = key.split('/', 2)
    index = ('labeled-data', folder, image_name)
    expected = frame['points']
    complete += all(expected[part] is not None for part in focus_parts)
    table = tables[folder]
    if index not in table.index:
        assert all(expected[part] is None for part in focus_parts), key
        continue
    for part in focus_parts:
        for coord, position in (('x', 0), ('y', 1)):
            actual = table.at[index, ('Kobe', part, coord)]
            point = expected[part]
            assert (pd.isna(actual) if point is None else abs(actual - point[position]) < 1e-6), (key, part, coord)
print(f'Validated {len(images)} images, {len(folders)} folders, and all focus labels; {complete} complete frames.')
print('Working project:', PROJECT)


In [ ]:
assert not (PROJECT / 'training-datasets').exists(), 'Training dataset already exists in this run. Do not overwrite it.'
SHUFFLE = 1
deeplabcut.create_training_dataset(
    str(CONFIG), Shuffles=[SHUFFLE], net_type='resnet_50', engine=deeplabcut.Engine.PYTORCH
)
datasets = list((PROJECT / 'training-datasets').rglob('CollectedData_Kobe.h5'))
assert len(datasets) == 1, datasets
print('Training dataset:', datasets[0])
print('Labeled rows:', len(pd.read_hdf(datasets[0])))


In [ ]:
assert torch.cuda.is_available(), 'GPU disconnected; reconnect before training.'
assert not list((PROJECT / 'dlc-models-pytorch').rglob('snapshot-*.pt')), 'Checkpoints already exist. Resume instead of starting this cell again.'
print('Training in Drive-backed project:', PROJECT)
deeplabcut.train_network(str(CONFIG), shuffle=SHUFFLE, save_epochs=10, epochs=200, batch_size=8)
print('Training finished. Checkpoints are under', PROJECT / 'dlc-models-pytorch')


In [ ]:
assert list((PROJECT / 'dlc-models-pytorch').rglob('snapshot-*.pt')), 'No trained checkpoint found.'
deeplabcut.evaluate_network(str(CONFIG), snapshotindex='all', per_keypoint_evaluation=True, plotting=False)
results = sorted((PROJECT / 'evaluation-results-pytorch').rglob('CombinedEvaluation-results.csv'))
assert results, 'Evaluation ran, but no combined results CSV was found.'
print('Results saved to:', results[-1])
display(pd.read_csv(results[-1]))
print('Project and checkpoints remain in Google Drive:', PROJECT)


## Notes

- This run uses a fresh 95/5 split, so its metrics are **not directly comparable** with the old model's 37-frame test split. Inspect the per-keypoint CSV and visual results before judging improvement.
- The new Drive run folder is printed in the setup and validation cells. Save that path. Checkpoints persist there if Colab disconnects. To resume, use DeepLabCut's `snapshot_path` / resume instructions with the existing run's config rather than rerunning the copy or create-dataset cells.
- The ZIP and JSON are frozen. Website labels added after September 24, 2026 require a new export and training run.
